# 🐺 The Witcher 3: RAG Inference Pipeline

Интерактивный ноутбук для демонстрации работы системы **Retrieval-Augmented Generation (RAG)** по базе знаний игры **«Ведьмак 3: Дикая Охота»** (Fandom Wiki).

### 🌟 Архитектура поиска и генерации:
1. **Query Expansion & Category Guidance:** Генерация синонимичных переформулировок запроса и тематических рубрик через LLM.
2. **Dense Multi-Query Retrieval:** Векторный поиск кандидатов в Milvus по нормализованным эмбеддингам `BAAI/bge-m3` (HNSW COSINE).
3. **Deduplication & Cross-Encoder Reranking:** Устранение дубликатов и глубокое переранжирование кросс-энкодером `BAAI/bge-reranker-large`.
4. **Prompt Augmentation & Generation:** Формирование контекста с цитированием первоисточников и генерация ответа через OpenAI-совместимый LLM сервер (vLLM, Ollama, LM Studio).

## 1. Настройка окружения и параметров

In [ ]:
import os
import logging
from pprint import pprint

# Импорт централизованной конфигурации проекта
from rag_config import settings

print("Параметры окружения:")
print(f"• Milvus URI:       {settings.milvus_uri}")
print(f"• Коллекция:        {settings.COLLECTION_NAME}")
print(f"• Модель эмбеддингов:{settings.EMBEDDING_MODEL}")
print(f"• Модель реранкера: {settings.RERANKER_MODEL}")
print(f"• LLM Сервер:       {settings.OPENAI_API_BASE}")
print(f"• LLM Модель:       {settings.LLM_MODEL}")

## 2. Инициализация RAG Пайплайна
Создаем экземпляр `WitcherRAGPipeline`, который подключается к Milvus, загружает модели `bge-m3` и `bge-reranker-large`, и подготавливает цепочку генерации.

In [ ]:
from rag_inference import WitcherRAGPipeline

# Инициализация пайплайна
pipeline = WitcherRAGPipeline()
print("✅ Пайплайн успешно инициализирован и готов к работе!")

## 3. Выполнение запросов к базе знаний

In [ ]:
query = "Что такое Дикая Охота и кто является её предводителем?"
result = pipeline.ask(query)

print(f"Вопрос: {query}\n")
print(f"Ответ:\n{result['answer']}\n")
print("Источники:")
for src in result["sources"]:
    print(f"  • {src}")

if result["categories"]:
    print(f"\nКатегории: {', '.join(result['categories'])}")

In [ ]:
query = "Какие ведьмачьи знаки существуют и для чего они нужны?"
result = pipeline.ask(query)

print(f"Вопрос: {query}\n")
print(f"Ответ:\n{result['answer']}\n")
print("Источники:")
for src in result["sources"]:
    print(f"  • {src}")

## 4. Диагностика: извлеченные документы и оценки реранкера

In [ ]:
# Получение сырых документов из ретривера
docs = pipeline.retriever.invoke("Кто такой Цири и каково её предназначение?")

print(f"Отобрано документов ретривером: {len(docs)}\n")
for i, doc in enumerate(docs, start=1):
    print(f"[{i}] 📘 {doc.metadata.get('title')} ({doc.metadata.get('subcategory')})")
    print(f"    🔗 {doc.metadata.get('url')}")
    print(f"    📝 {doc.page_content[:250]}...\n")